<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 8.2: Working with Text

The purpose of the lab is to see how the [spaCy](https://spacy.io) and [nltk](https://www.nltk.org/) libraries can be used to perform the following text processing tasks:
- tokenisation
- stemming and lemmatisation
- part of speech tagging
- named entity recognition

We will also see how such processing affects word frequency analysis.

INSTRUCTIONS:
- Run the cells
- Observe and understand the results
- Answer the questions indicated and in the cells labelled "ANSWER"

## Import libraries

In [ ]:
## Import Libraries
%matplotlib inline
import matplotlib.pyplot as plt
from collections import Counter

import regex as re
import spacy

## Load data

In [ ]:
## Loading the data

input_file = '../../data/ncc-1701-D.txt' # Text related to "Star Trek"

with open(input_file, 'r') as f:
    text = f.read()

## Inspect the data

In [ ]:
# ANSWER - use the print function to display the text


## Preprocess the data
- Remove quotes (") and characters around headings (=)
- Reassign the cleaned text to the variable `text` and print the text

Further examples of text preprocessing include the following (NOT to be done here):

- making all text lowercase
- removing or converting special symbols, e.g. "é" to "e"
- removing or converting emoticons
- removing tags (e.g., &lt;p&gt;, &lt;li&gt;) from web-scraped data
- converting non-UTF-8 characters to standard UTF-8 format
- deleting URLs, email addresses, social media handles or hashtags
- contraction expansion (e.g. I'll -> I will)

In [ ]:
# ANSWER


## Helper method

The following function to be used later creates a bar chart of the frequency of words in the text.

In [ ]:
def plot_words(tokens, top = 30):
    tokens_counter = Counter(tokens)
    tok = [t for (t, _) in tokens_counter.most_common()]
    val = [v for (_, v) in tokens_counter.most_common()]

    plt.figure(figsize = (16, 6))
    plt.bar(tok[:top], val[:top])
    plt.title('Number of terms: %d' % len(tokens_counter))
    plt.xticks(rotation = 90)

    for i, label in enumerate(val[:top]):
        plt.text(i, label, label, ha='center', va='bottom')

    plt.show()

## Text processing with spaCy
spaCy for text processing and understanding requires the loading of an English model.

In [ ]:
# load spaCy and the English model
!python -m spacy download en_core_web_sm
nlp = spacy.load('en_core_web_sm')

# process the text
doc = nlp(text)

### Tokenise

In [ ]:
type(doc)

spacy.tokens.doc.Doc

Note that `doc` is a sequence of tokens that can be iterated over:

In [ ]:
for i, t in enumerate(doc):
    print('%2d| %r' % (i+1, t.text))

    # break at the end of the first sentence:
    if t.text == '.':
        break

The `plot_words` function can be used in the following way on the tokens in `doc`:

In [ ]:
plot_words(['%r' % t.text for t in doc])

### Stop Words

Note from the following cell what stop words are removed from the first sentence.
(Would you expect the word 'five' to be a stop word?)

In [ ]:
print('i | with stop words without')
print('--| --------------- ------------')

# for all the tokens
for i, t in enumerate(doc):
    print('%2d| %-15r %r' % (i+1, t.text, ('' if t.is_stop else t.text)))

    # break after the first sentence
    if t.text == '.':
        break

Call the `plot_words` function on the list of tokens from `doc` which are neither stop words (`t.is_stop`) nor punctuation token (`t.is_punct`).

In [ ]:
# ANSWER


### Check Part of Speech

Run the following cell to see the parts of speech and tag of each token.

In [ ]:
## spaCy
for i, t in enumerate(doc):
    print('%2d|%-12r : %-5s %s' % (i+1, t.text, t.pos_, t.tag_))
    if t.text == '.':
        break

### Lemmatisation

`t.lemma_` gives the lemma form of a token in spaCy.

In [ ]:
## spaCy
print('i | Token        Lemma')
print('--| ------------ ------------')
for i, t in enumerate(doc):
    print('%2d| %-12r %r' % (i+1, t.text, t.lemma_))
    if t.text == '.':
        break

Run the `plot_words` function on the lemma forms of each token that are neither stop words nor punctuation tokens.

In [ ]:
# ANSWER


### Identify Entities

Observe how the following code iterates over each sentence and shows the named entities found in each one.

In [ ]:
## spaCy
for i, s in enumerate(doc.sents):
    print('%2d: %s' % (i, re.sub(r'\n+', '', s.text)))
    if s.as_doc().ents:
        print('-'*80)
        for e in s.as_doc().ents:
            print('%-11s: %s' % (e.label_, re.sub(r'\n+', '', e.text)))
    print('='*80)

## Text processing with NLTK

In this section we carry out the above tasks with the NLTK library - note any differences in outputs.

In [ ]:
import nltk

### Tokenise

In [ ]:
nltk.download('punkt_tab')

In [ ]:
# use nltk to find tokens
tokens = nltk.word_tokenize(text)

for i, t in enumerate(tokens[:25]):
    print('%2d| %r' % (i+1, t))

The code may give a few LookupErrors to begin with. Simply follow the nltk.download steps as indicated.

Run `plot_words` on the `tokens` variable created above.

In [ ]:
# ANSWER


### Stop Words

In [ ]:
nltk.download('stopwords')
stopWords = nltk.corpus.stopwords.words('english')

stopWords.sort()
print(', '.join(stopWords))

In [ ]:
# create a list of tokens withOUT the stop words
# NOTE: see the `.lower()` method applied to token
tokens_no_stop = [t for t in tokens if t.lower() not in stopWords]

In [ ]:
## NLTK
i = 0
j = 0

print('i | with stop words without')
print('--| --------------- ------------')

# for all the tokens
while i < len(tokens):
    # same word
    if tokens[i] == tokens_no_stop[j]:
        print('%2d| %-15r %r' % (i+1, tokens[i], tokens_no_stop[j]))
        j += 1
    # not the same word
    else:
        print('%2d| %-15r' % (i+1, tokens[i]))

    # next word
    i += 1
    # break after the first sentence
    if tokens[i-1] == '.':
        break

Run the `plot_words` function on the list of tokens without stop words:

In [ ]:
# ANSWER


### Parts of Speech

In [ ]:
# define PoS list
pos_list = {
    'CC':   'coordinating conjunction',
    'CD':   'cardinal digit',
    'DT':   'determiner',
    'EX':   'existential there (like: "there is" ... think of it like "there exists")',
    'FW':   'foreign word',
    'IN':   'preposition/subordinating conjunction',
    'JJ':   'adjective "big"',
    'JJR':  'adjective, comparative "bigger"',
    'JJS':  'adjective, superlative "biggest"',
    'LS':   'list marker 1)',
    'MD':   'modal could, will',
    'NN':   'noun, singular "desk"',
    'NNS':  'noun plural "desks"',
    'NNP':  'proper noun, singular "Harrison"',
    'NNPS': 'proper noun, plural "Americans"',
    'PDT':  'predeterminer "all the kids"',
    'POS':  'possessive ending parent"s',
    'PRP':  'personal pronoun I, he, she',
    'PRP$': 'possessive pronoun my, his, hers',
    'RB':   'adverb very, silently,',
    'RBR':  'adverb, comparative better',
    'RBS':  'adverb, superlative best',
    'RP':   'particle give up',
    'TO':   'to go "to" the store.',
    'UH':   'interjection errrrrrrrm',
    'VB':   'verb, base form take',
    'VBD':  'verb, past tense took',
    'VBG':  'verb, gerund/present participle taking',
    'VBN':  'verb, past participle taken',
    'VBP':  'verb, sing. present, non-3d take',
    'VBZ':  'verb, 3rd person sing. present takes',
    'WDT':  'wh-determiner which',
    'WP':   'wh-pronoun who, what',
    'WP$':  'possessive wh-pronoun whose',
    'WRB':  'wh-abverb where, when',
}

In [ ]:
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
## nltk
tagged = nltk.pos_tag(tokens)

for i, t in enumerate(tagged[:25]):
    print('%2d|%-12r : %-4s %s' % (i+1, t[0], t[1], (pos_list[t[1]] if t[1] in pos_list else '-')))

### Stemming

In [ ]:
## nltk
ps = nltk.porter.PorterStemmer()
stemmed = ' '.join([ps.stem(word) for word in text.split()])

### Lemmatisation

In [ ]:
nltk.download('wordnet')

In [ ]:
## nltk
wl = nltk.stem.WordNetLemmatizer()
lemma = ' '.join([wl.lemmatize(word) for word in text.split()])

In [ ]:
## nltk
dot = stemmed.find('.') + 1
sl = stemmed[:dot].split()
dot = lemma.find('.') + 1
ll = lemma[:dot].split()

print('i | Stem           Lemma')
print('--| -------------- ------------')
for i, p in enumerate(zip(sl, ll)):
    print('%2d| %-12r   %-12r' % (i+1, p[0], p[1]))

Run the `plot_words` function on the stemmed and lemma forms of tokens (hint: note the difference between calling the function on `stemmed` and `stemmed.split()`).

In [ ]:
# ANSWER


In [ ]:
# ANSWER


### Identify Entities

In [ ]:
nltk.download('maxent_ne_chunker_tab')
nltk.download('words')

In [ ]:
## Identify entities
entities = nltk.chunk.ne_chunk(tagged)

for e in entities:
    s = re.sub(r'[\(\)]', '', str(e))
    if s.find('/NNP') > 0:
        t = s.split()[0]
        n = ' '.join([re.sub(r'/NNP', '', x) for x in s.split()[1:]])
        print('%-12s: %s' % (t, n))

## Conclusion
We used spaCy and NLTK for basic text preprocessing and saw the impact on word-frequency analysis. This makes text more useful, structured, and informative for downstream tasks such as classification, search or clustering, or modelling.



---



---



> > > > > > > > > © 2026 Institute of Data


---



---



